# DeltaConv : audit CPU des bases tangentielles

Ce notebook vérifie les conventions numériques d'un gradient MLS, **pas un réseau entraîné**. Il lit les mesures
de référence et permet de relancer les 288 calculs sur deux surfaces analytiques. Aucun GPU ni téléchargement
de données n'est requis. Les quatre variantes sont détaillées dans `docs/research.md`.

Le clonage Colab demande que le dépôt cible soit publié. Localement, ouvrir ce notebook depuis le projet.

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nassim-arifette/deltaconv-point-clouds.git"
here = Path.cwd()
project = next((p for p in (here, here.parent) if (p / "deltaconv_audit").is_dir()), None)
if project is None:
    if not IN_COLAB:
        raise RuntimeError("Run this notebook from the project or its notebooks directory.")
    project = Path("/content/deltaconv-point-clouds")
    if not project.exists():
        subprocess.run(["git", "clone", REPO_URL, str(project)], check=True)
    if not (project / "deltaconv_audit").is_dir():
        raise RuntimeError("The published project checkout is missing. Upload the project folder instead.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
print("Project:", project)

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.4.6"], check=True)
import json
reference = json.loads((project / "results/cpu_reference/results.json").read_text(encoding="utf-8"))
print("Type de preuve:", reference["evidence_kind"])
print("Conditions:", len(reference["rows"]), "| statut:", reference["status"])
for row in reference["summary"]:
    print(row["method"], "erreur gradient =", row["gradient_relative_l2_mean"],
          "dérive de base =", row["gauge_normalized_relative_l2_mean"])

## Relancer avec les paramètres enregistrés

Les normales exactes et PCA sont évaluées séparément. Chaque paire avant/après rotation utilise le même signal
et les mêmes voisins. Les données sont synthétiques : elles possèdent un gradient analytique de référence.
Les sorties sont placées dans un nouveau dossier pour conserver les mesures précédentes.

In [ ]:
from datetime import datetime, timezone
POINTS = 512  #@param {type:"integer"}
output = project / "results/runs" / datetime.now(timezone.utc).strftime("notebook-%Y%m%d-%H%M%S")
subprocess.run([sys.executable, "-m", "deltaconv_audit.experiment", "--output", str(output),
                "--points", str(POINTS), "--seeds", "0", "1", "2"], cwd=project, check=True)
measured = json.loads((output / "results.json").read_text(encoding="utf-8"))
print("Mesures et provenance:", output)

## Examiner une condition sans confondre les moyennes

L'erreur au gradient analytique est mesurée avant normalisation. La dérive au changement de bases est mesurée
avant et après normalisation. Le filtre ci-dessous porte sur les régions réellement sous-échantillonnées.

In [ ]:
rows = [r for r in measured["rows"] if r["density"] == "biased" and r["normal_source"] == "pca"
        and r["sigma"] == 0.02 and r["seed"] == 0]
for row in rows:
    print(row["shape"], row["method"], "erreur globale:", row.get("gradient_relative_l2"),
          "erreur zone rare:", row.get("sparse_gradient_relative_l2"),
          "dérive base:", row.get("gauge_normalized_relative_l2"))

## Interprétation

Une dérive faible indique que le choix des coordonnées ne modifie plus l'opérateur dans ces tests. Elle ne
prouve pas un gain de classification. La variante locale peut améliorer une région et dégrader l'ensemble ;
conserver ces deux résultats. Le second notebook compare la baseline NumPy au vrai code amont avant toute
étude du réseau, dans un environnement différent.